처음 한 번만 실행

mkdir -p models

python -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='unsloth/gemma-3-27b-it-GGUF', filename='gemma-3-27b-it-UD-Q6_K_XL.gguf', local_dir='./models', local_dir_use_symlinks=False)"

매 번 실행

export LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH

./llama.cpp/build/bin/llama-server \
  -m ./models/gemma-3-27b-it-UD-Q6_K_XL.gguf \
  -c 25000 \
  -np 4 \
  -cb \
  -fa on \
  --port 8000 \
  --host 0.0.0.0

In [ ]:
import asyncio
import ast
import pandas as pd
from openai import AsyncOpenAI

model_name = "Gemma3-27B"
df_test = pd.read_csv('test_with_class.csv')
df_output = pd.read_csv('output.csv')

client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="sk-no-key-required",
)

In [ ]:
system_msg_0_0 = """ 과목의 문제가 주어집니다. 문제의 제시문, 질문, 객관식 선택지 속에 적혀있는 용어들 중에서, 이 문제를 해결하기 위해 지문에 명시된 내용 외의 관련정보를 자세히 알아야 하는 """
system_msg_0_1 = """ 과목의 전문적인 용어들을 모아서 하나의 배열 형태로 출력해주세요. 용어가 해당 과목에 속하는 용어이면서 난이도가 있는 개념인 경우에는 반드시 배열에 포함합니다. 또한 특정한 인물, 단체, 사건, 물건, 이론, 시기, 장소 등과 같이 해당 과목에서 중요한 구체적 대상이 문제의 제시문, 질문, 객관식 선택지 속에 등장하는 경우에도 배열에 반드시 포함합니다. 그러나 용어가 해당 과목의 전문용어가 아니라 일상어로도 분류되기도 할 만큼 보편적인 단어인 경우에는 배열에 넣지 않도록 합니다."""

system_msg_1_0 = """ 분야의 문제가 주어집니다. 문제의 제시문, 질문, 객관식 선택지 속에 적혀있는 용어들 중에서, 이 문제를 해결하기 위해 지문에 명시된 내용 외의 관련정보를 자세히 알아야 하는 """
system_msg_1_1 = """ 분야의 전문적인 용어들을 모아서 하나의 배열 형태로 출력해주세요. 용어가 해당 분야에 속하는 용어이면서 난이도가 있는 개념인 경우에는 반드시 배열에 포함합니다. 또한 특정한 인물, 단체, 사건, 물건, 이론, 시기, 장소 등과 같이 해당 과목에서 중요한 구체적 대상이 문제의 제시문, 질문, 객관식 선택지 속에 등장하는 경우에도 배열에 반드시 포함합니다. 그러나 용어가 해당 분야의 전문용어가 아니라 일상어로도 분류되기도 할 만큼 보편적인 단어인 경우에는 배열에 넣지 않도록 합니다."""


list_system_msg = [(system_msg_0_0, system_msg_0_1), (system_msg_1_0, system_msg_1_1)]

In [ ]:
async def get_inference(system_msg, user_content, seed):
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_content}
            ],
            max_tokens=2048,
            temperature=1.0,
            top_p=0.95,
            seed=seed,
            extra_body={"min_p": 0.0, "top_k": 30}
        )
        ret = response.choices[0].message.content
        print(system_msg)
        print(ret)
        print()
        return ret

    except Exception as e:
        print(f"Inference {seed} Error: {e}")
        return "Error"


async def process_row(index, row, cls, seed_base):
    problem = ast.literal_eval(row['problems'])
    user_content = f"<제시문>\n{row['paragraph']}\n\n"
    if pd.notna(row['question_plus']):
        user_content += f"<보기>\n{row['question_plus']}\n\n"
    user_content += f"<질문>\n{problem['question']}\n"
    for k in range(len(problem['choices'])):
        user_content += f"{k+1}. {problem['choices'][k]}\n"

    seed = (seed_base * len(df_test) + index)

    tasks = []
    for r in range(len(list_system_msg)):
        tasks.append(get_inference(f"'{cls}'" + list_system_msg[r][0] + f"'{cls}'" + list_system_msg[r][1], user_content, seed))
    
    results = await asyncio.gather(*tasks)
    
    return index, results


async def main():
    print(f"Start Extraction with Model: {model_name}")

    for i in range(0, 869):
        list_class = []
        for k in range(4):
            list_class.append(df_test.loc[i, f'class_{k}'])
            
        if "국어" in list_class or "문학" in list_class:
            continue
        
        print(f"Processing i={i} (Parallel requests for {len(list_system_msg)} personas)...")
        
        for k in range(len(list_class)):
            idx, results = await process_row(i, df_test.loc[i], list_class[k], 0)
            for r, output in enumerate(results):
                df_test.loc[idx, f'keyword_{k}_{r}'] = output.strip()
        
        df_test.to_csv(f'TestSet_ExtractKeyWord_{model_name}.csv', index=False)
    df_test.to_csv(f'TestSet_ExtractKeyWord_{model_name}.csv', index=False)

In [ ]:
await main()